# 🚀 DKV Remote CUDA Benchmark & Telemetry Runner

This notebook coordinates the compilation, execution, telemetry sweep, and secure cleanup for **DKV Active**, **DKV Native**, and **Dense Baseline** configurations on a remote Linux GPU machine (RTX card).

### Privacy Guard: Isolated Sandbox
All downloaded weights, compiled Triton caches, and PyTorch JIT compilation files are redirected to a temporary workspace directory (`./remote_cache`) within this repository. This prevents leakage into system directories like `~/.cache/huggingface` or `~/.cache/triton` and ensures that running the **Secure Cleanup** at the bottom will completely purge any trace of your unpublished paper/code from this machine.

## 🛠️ Step 1: Environment Diagnostics
Let's confirm the CUDA toolkit, PyTorch versions, and GPU capabilities on this machine.

In [ ]:
import sys
import os
import subprocess
import torch

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Compute Capability: {torch.cuda.get_device_capability(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

try:
    nvcc_ver = subprocess.check_output(["nvcc", "--version"]).decode("utf-8").split("\n")[-2]
    print(f"NVCC Version: {nvcc_ver}")
except Exception:
    print("⚠️ nvcc command not found in PATH. Make sure the CUDA Toolkit is installed and added to your environmental variables.")

## ⚙️ Step 2: Auto-Compilation of Engines
We compile the CPython CUDA Active extension and the C++ Native executable. This will take about 1–2 minutes.

In [ ]:
!python remote_cuda_benchmark.py --compile

## 📈 Step 3: Run Telemetry Sweeps
Run the benchmark sweeps. This downloads the model weights (into `./remote_cache`) and sweeps through specified context lengths. 

You can customize:
- `--model-id`: Hugging Face model for Active & Dense (default: `Qwen/Qwen2.5-1.5B-Instruct`)
- `--gguf-repo` & `--gguf-file`: GGUF model for Native (default: `Qwen/Qwen2.5-1.5B-Instruct-GGUF`)
- `--contexts`: List of context lengths (e.g. `4096 8192 16384 32768`)
- `--gen-len`: Generation token budget (default: `32`)

*Note: For 7B / 13B / 8B tests, change the parameters below to Llama-3 or Qwen2.5-7B.*

In [ ]:
# Run the sweep (4k, 8k, 16k context lengths by default)
!python remote_cuda_benchmark.py \
    --model-id "Qwen/Qwen2.5-1.5B-Instruct" \
    --gguf-repo "Qwen/Qwen2.5-1.5B-Instruct-GGUF" \
    --gguf-file "qwen2.5-1.5b-instruct-q4_k_m.gguf" \
    --contexts 4096 8192 16384 \
    --gen-len 32 \
    --output benchmark_results.json

## 📊 Step 4: Visualize Results & Telemetry
Let's load the telemetry results, print a comparison table, and plot the stats (Prefill Latency, TTFT, Decode TPS, Peak VRAM).

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

results_file = "../benchmark_results.json" if os.path.exists("../benchmark_results.json") else "benchmark_results.json"

with open(results_file) as f:
    data = json.load(f)
df = pd.DataFrame(data)

# Display summary table
print("\n=== Telemetry Comparison Table ===")
summary_tbl = df.pivot(index="ctx_target", columns="engine", values=["prefill_s", "ttft_s", "decode_tps", "peak_vram_gb"])
print(summary_tbl.to_markdown() if hasattr(summary_tbl, "to_markdown") else summary_tbl)

# Create comparison graphs
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("DKV Active vs Native vs Dense Baseline on CUDA", fontsize=16, fontweight="bold")

metrics = [
    ("prefill_s", "Prefill Time (Seconds) - Lower is Better"),
    ("ttft_s", "Time to First Token (Seconds) - Lower is Better"),
    ("decode_tps", "Decode Throughput (Tokens/Sec) - Higher is Better"),
    ("peak_vram_gb", "Peak VRAM Usage (GB) - Lower is Better")
]

for idx, (metric, ylabel) in enumerate(metrics):
    ax = axes[idx // 2, idx % 2]
    sns.barplot(data=df, x="ctx_target", y=metric, hue="engine", ax=ax, palette="viridis")
    ax.set_title(ylabel, fontweight="semibold")
    ax.set_xlabel("Context Length (Tokens)")
    ax.set_ylabel(metric.split("_")[0].upper())

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig("telemetry_graphs.png", dpi=300)
plt.show()

## 🧹 Step 5: Secure Destruct & Cleanup
Run this cell when you are ready to log out of the remote PC. 

**What this does:**
1. Deletes the entire `./remote_cache` folder (containing HF model downloads, GGUF weights, and Triton compilation databases).
2. Deletes all compiled Active shared libraries (`.so`) and C++ build folders.
3. Keeps only your `benchmark_results.json` and `telemetry_graphs.png` so you can retrieve your statistics.

In [ ]:
# Wipe everything securely
!python remote_cuda_benchmark.py --cleanup